In [1]:
# =============================================================================
# STEP 0: Force reload modules (run this first after code changes!)
# =============================================================================
import importlib
import src.data_loader
import src.features
import src.models.base
import src.models.goals
import src.models.assists
import src.models.minutes
import src.models.defcon
import src.models.clean_sheet
import src.models.bonus
import src.models.cards
import src.models.saves
import src.pipeline
import src.viz

importlib.reload(src.data_loader)
importlib.reload(src.features)
importlib.reload(src.models.base)
importlib.reload(src.models.goals)
importlib.reload(src.models.assists)
importlib.reload(src.models.minutes)
importlib.reload(src.models.defcon)
importlib.reload(src.models.clean_sheet)
importlib.reload(src.models.bonus)
importlib.reload(src.models.cards)
importlib.reload(src.models.saves)
importlib.reload(src.pipeline)
importlib.reload(src.viz)

print("All modules reloaded.")

All modules reloaded.


In [2]:
# =============================================================================
# STEP 1: Update Data (optional - only if you need fresh gameweek data)
# =============================================================================
!python scrape_update_data.py --gameweek 31
#!python scrape_update_data.py --auto

FotMob Incremental Data Updater
Launching browser to solve Cloudflare challenge...
Session established! (17 cookies)

Season: 2025/2026
Fetching fixture list...
Found 309 completed matches in 2025/2026
Checking existing data in: player_stats.csv
Found 2958 existing match IDs in data
Resolving FPL gameweeks [31] via FPL API...
  FPL API returned 8 fixtures for GW [31]
  Matched 8 FotMob fixtures
  8 new matches to scrape
Matches per matchday: {31: 8}
[1/8] MD31: AFC Bournemouth vs Manchester United (30 players)
[2/8] MD31: Brighton & Hove Albion vs Liverpool (31 players)
[3/8] MD31: Fulham vs Burnley (31 players)
[4/8] MD31: Everton vs Chelsea (31 players)
[5/8] MD31: Leeds United vs Brentford (26 players)
[6/8] MD31: Newcastle United vs Sunderland (31 players)
[7/8] MD31: Aston Villa vs West Ham United (31 players)
[8/8] MD31: Tottenham Hotspur vs Nottingham Forest (32 players)
Session closed.

Saving data...
Saved: data\players\player_stats.csv (84009 total records)
Saved: data\matche

In [3]:
# =============================================================================
# STEP 2: Run the Pipeline
# =============================================================================
from src.pipeline import FPLPipeline
from src.experiment_log import clear_experiments

pipeline = FPLPipeline('data', n_sims=5000)
pipeline.load_data()
pipeline.compute_features()

# Clear old experiments (old MAE was not using 25/26 actuals with bonus)
clear_experiments('data')
print("Cleared old experiment history (incompatible MAE calculation)")

# Full hyperparameter tuning (all models, 2025/26 holdout)
pipeline.tune(
    n_iter=100,
    use_subprocess=True,
    test_season='2025/2026',
    description='baseline: full tune, 5k sims, 25/26 holdout inc-bonus MAE',
    test_start_gw=20,
)

# Train final models on all data
pipeline.train()

# Predict
predictions = pipeline.predict(gameweek=32, season='2025/2026')

LOADING DATA
Loading player stats from: player_stats.csv
  Loaded 84,009 player-match records
  Seasons: ['2018/2019', '2019/2020', '2020/2021', '2021/2022', '2022/2023', '2023/2024', '2024/2025', '2025/2026']
Loaded 2,970 fixtures
Filtered to seasons: ['2020/2021', '2021/2022', '2022/2023', '2023/2024', '2024/2025', '2025/2026']
Current season (2025/2026): 521 active players
Final dataset: 63,736 records
Fetching actual FPL points for GW 1-31 (31 GWs)...
  Cache has 31 GWs already
  Cached to data\fpl_actual_points.csv
  Total: 23,911 player-gameweek records
  FPL card merge: 5,630 rows matched, 5,630 with card data, 407 total yellows, 42,940 DGW rows excluded

COMPUTING FEATURES
Computing rolling features...


c:\Users\dpfin\OneDrive\Desktop\projecting_fpl_v2\src\features.py:136: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f'fouls_committed_per90_roll{window}'] = df.groupby('player_id')['fouls_committed_per90'].transform(
c:\Users\dpfin\OneDrive\Desktop\projecting_fpl_v2\src\features.py:136: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f'fouls_committed_per90_roll{window}'] = df.groupby('player_id')['fouls_committed_per90'].transform(
c:\Users\dpfin\OneDrive\Desktop\projecting_fpl_v2\src\features.py:136: PerformanceWarning: D

  Computed 288 rolling/lifetime features
Cleared old experiment history (incompatible MAE calculation)

TUNING HYPERPARAMETERS WITH HOLDOUT TEST SET

Data split (season=2025/2026):
  Train: 60,160 samples (['2020/2021', '2021/2022', '2022/2023', '2023/2024', '2024/2025', '2025/2026'])
  Test:  3,563 samples (['2025/2026'])
    2025/2026: GW20.0-GW31.0

------------------------------------------------------------
PHASE 1: Hyperparameter + Feature Selection Tuning (5-fold CV)
------------------------------------------------------------

Tuning MINUTES (two-stage, 100 trials each)...
  Total played: 60,160

  [1/3] StarterClassifier (75 features, 71.5% starters)
    Pre-computing feature rankings...


C:\Users\dpfin\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Best trial: 85. Best value: 0.441679: 100%|██████████| 100/100 [03:20<00:00,  2.01s/it]


    Best CV LogLoss: 0.4417
    Feature method: xgb_gain, selected 26/75 features

  [2/3] StarterMinutesModel (31 features, 43,043 samples)
    Pre-computing feature rankings...


...


Model           Metric          Test Value   MAE         
------------------------------------------------------
MINUTES         Huber Loss      142.8566     17.6214     
  Classifier    AUC             0.8117       LogLoss 0.7456
  Starter Reg   MAE             11.8026     
  Sub Reg       MAE             30.7535     
GOALS           Poisson Dev     0.3715       0.1512      
ASSISTS         Poisson Dev     0.3327       0.1166      
DEFCON          Poisson Dev     2.2971       2.7492      
SAVES           MAE             1.3958       1.3958      
CLEAN_SHEET     Poisson Dev     1.1209       0.8903      
------------------------------------------------------
FPL POINTS      ex-bonus MAE    1.7310      
                inc-bonus MAE   1.8674      
BONUS           MAE             0.2812      

(Lower is better for all metrics)
ex-bonus: FPL points excluding bonus on both sides
inc-bonus: FPL points with bonus on both sides (pred bonus vs actual bonus)

Experiment logged: 20260323_235005_

C:\Users\dpfin\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\xgboost\core.py:158: UserWarning: [23:50:05] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-08cbc0333d8d4aae1-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


  StarterMinutesModel: 45,512 samples, mean=85.9
  SubMinutesModel: 18,211 samples, mean=21.9
  Combined MAE: 15.3
Training CleanSheetModel (Goals Against) on 4,020 team-matches (50 features, tuned selection)...
  Avg goals conceded: 1.375
  Actual CS rate: 27.6%
  Prior lambda range: 0.30 - 4.00
  MAE (goals against): 0.929
  Predicted CS prob (mean): 30.3%
  Predicted 2+ conceded prob (mean): 37.4%
  Predicted CS prob range: 2.5% - 72.9%

Generating leak-free predicted team goals (TimeSeriesSplit OOF)...


c:\Users\dpfin\OneDrive\Desktop\projecting_fpl_v2\src\pipeline.py:279: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  self.df['opponent_norm'] = self.df['opponent'].apply(normalize_name)


  Mapped pred_team_goals to 63,736/63,736 player rows
  Mean pred_team_goals: 1.393
Training GoalsModel on 63,723 samples (57 features, tuned selection)...
  Target mean: 0.0950
  MAE: 0.1557
Training AssistsModel on 63,723 samples (57 features, tuned selection)...
  Target mean: 0.0677
  MAE: 0.1246
Training DefconModel on 63,723 samples (46 features, tuned selection)...
  Target mean: 5.8963
  MAE: 2.3123
  NB dispersion r=18.29 (var/mean ≈ 1.35x)
Training BonusModel (Monte Carlo, 5000 sims)...
  Estimating baseline BPS from stats (no actual BPS data)
Training BaselineBPSModel on 45512 samples...
  Features used: 32
  Avg baseline BPS: 18.7
  MAE: 0.69
  Loaded FPL availability for 1618 players
Training CardsModel on 63,723 samples (50 features)...
  Yellow cards: 407/63,723 (0.6%)
  LogLoss: 0.0202
Training SavesModel on 4,413 GK samples (37 features, tuned selection)...
  Mean saves/90: 3.00
  MAE: 1.464

PREDICTING GW32 (2025/2026)
Found 521 active players with historical data
  F

In [4]:
# =============================================================================
# STEP 3: View Top Players
# =============================================================================
# Top 20 by expected points with full prediction breakdown
display_cols = [
    'player_name', 'team', 'fpl_position', 'opponent', 'is_home',
    'pred_minutes', 'pred_exp_goals', 'pred_exp_assists', 
    'pred_cs_prob', 'pred_2plus_conceded', 'pred_4plus_conceded',
    'pred_defcon_prob', 'pred_exp_saves',
    'pred_yellow_prob', 'pred_red_prob',
    'pred_bonus', 'exp_total_pts'
]
available_cols = [c for c in display_cols if c in predictions.columns]
predictions.loc[predictions['fpl_position']!="GK"].nlargest(20, 'exp_total_pts')[available_cols].round(2)

,player_name,team,fpl_position,opponent,is_home,pred_minutes,pred_exp_goals,pred_exp_assists,pred_cs_prob,pred_2plus_conceded,pred_4plus_conceded,pred_defcon_prob,pred_exp_saves,pred_yellow_prob,pred_red_prob,pred_bonus,exp_total_pts
54,Bruno Fernandes,Manchester United,MID,Leeds,1,89.610001,0.39,0.39,0.44,0.20,0.01,0.15,0.0,0.00,0.00,0.70,6.53
299,Cole Palmer,Chelsea,MID,Man City,1,84.379997,0.50,0.19,0.32,0.31,0.03,0.03,0.0,0.00,0.00,0.64,6.09
40,Mohamed Salah,Liverpool,MID,Fulham,1,88.349998,0.46,0.22,0.39,0.25,0.02,0.01,0.0,0.00,0.00,0.71,6.08
228,Bryan Mbeumo,Manchester United,MID,Leeds,1,88.000000,0.41,0.23,0.44,0.20,0.01,0.01,0.0,0.04,0.00,0.51,5.69
139,Diogo Dalot,Manchester United,DEF,Leeds,1,88.440002,0.08,0.11,0.44,0.20,0.01,0.25,0.0,0.00,0.00,0.33,5.17
171,Cody Gakpo,Liverpool,MID,Fulham,1,84.680000,0.34,0.17,0.39,0.25,0.02,0.03,0.0,0.00,0.01,0.43,5.08
69,Jarrod Bowen,West Ham United,FWD,Wolves,1,88.940002,0.38,0.21,0.25,0.40,0.05,0.13,0.0,0.00,0.00,0.93,5.07
365,Dango Ouattara,Brentford,MID,Everton,1,85.190002,0.29,0.20,0.35,0.28,0.02,0.09,0.0,0.05,0.00,0.53,5.06
253,Bukayo Saka,Arsenal,MID,Bournemouth,1,88.120003,0.28,0.23,0.39,0.25,0.02,0.04,0.0,0.00,0.01,0.47,5.02
382,Igor Thiago,Brentford,FWD,Everton,1,88.989998,0.46,0.10,0.35,0.28,0.02,0.04,0.0,0.04,0.01,0.95,5.02


In [5]:
# =============================================================================
# Interactive Distribution (D3.js) — opens in browser
# =============================================================================
from src.viz import generate_distribution_html, generate_mobile_html
import webbrowser, os

# Get eval metrics from training
viz_metrics = pipeline.get_viz_metrics()

# Desktop version (ridge plot)
html_path = generate_distribution_html(
    predictions,
    pipeline.last_simulations,
    output_path='distributions.html',
    top_n=100,
    gameweek=32,
    metrics=viz_metrics,
)

# Mobile version (card layout)
mobile_path = generate_mobile_html(
    predictions,
    pipeline.last_simulations,
    output_path='distributions_mobile.html',
    top_n=100,
    gameweek=32,
    metrics=viz_metrics,
)

# Save full run (predictions, simulations, params, metrics)
run_dir = pipeline.save_run(
    predictions,
    gameweek=32,
    season='2025/2026',
    description='baseline: full tune, 5k sims, 25/26 holdout inc-bonus MAE',
)

webbrowser.open('file://' + os.path.realpath(html_path))

Distribution visualization saved to: distributions.html
Mobile distribution visualization saved to: distributions_mobile.html

Run saved to: data\runs\gw32_20260323_235026
  predictions.csv: 456 players
  simulations: 5000 sims
  tuned_params.json: 6 models
  FPL MAE: ex-bonus=1.7310, inc-bonus=1.8674


True

In [6]:
# =============================================================================
# FPL Points Breakdown: Predicted vs Actual by Category
# Shows where the model over/under-predicts in FPL points terms
# =============================================================================
breakdown = pipeline.points_breakdown()
display(breakdown.style.format({
    'pred_value_avg': '{:.4f}',
    'pred_pts_avg': '{:.4f}',
    'actual_value_avg': '{:.4f}',
    'actual_pts_avg': '{:.4f}',
    'pts_diff': '{:+.4f}',
    'abs_pts_diff': '{:.4f}',
}, na_rep='—').set_caption('FPL Points Breakdown: Predicted vs Actual (test set averages per player-match)'))

c:\Users\dpfin\OneDrive\Desktop\projecting_fpl_v2\src\pipeline.py:2163: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  result = pd.concat([result, totals], ignore_index=True)


,category,pred_value_avg,pred_pts_avg,actual_value_avg,actual_pts_avg,pts_diff,abs_pts_diff
0,Appearance,67.9708,1.6753,65.9829,1.6930,-0.0177,0.0177
1,Goals,0.0926,0.4302,0.0862,0.3938,+0.0365,0.0365
2,Assists,0.0655,0.1966,0.0620,0.1861,+0.0105,0.0105
3,Clean Sheet,0.2111,0.3317,0.2324,0.4942,-0.1626,0.1626
4,Goals Conceded,1.3945,-0.1500,1.2186,-0.1313,-0.0187,0.0187
5,Saves,2.9148,0.0657,2.7220,0.0393,+0.0264,0.0264
6,Defcon,0.2163,0.2095,0.2312,0.2240,-0.0144,0.0144
7,Yellow Cards,0.0303,-0.0303,0.0427,-0.0427,+0.0124,0.0124
8,Red Cards,0.0027,-0.0080,0.0022,-0.0067,-0.0013,0.0013
9,Bonus,0.2206,0.2206,0.0862,0.0862,+0.1344,0.1344
